# Quick-Start 02: Policy Gradient — REINFORCE on CartPole-v1

**Estimated time: under 2 minutes**

## Learning Objectives

By the end of this notebook you will:
1. Understand the core idea of policy gradient methods: directly optimise the policy with gradient ascent
2. Implement the REINFORCE algorithm using a small PyTorch neural network
3. Compare policy-gradient learning dynamics with the Q-learning approach from Quick-Start 01

## Prerequisites

- Quick-Start 00 and 01 (RL loop, Q-learning)
- Basic PyTorch (tensors, autograd, `optimizer.step()`)

## Dependencies

```
pip install gymnasium numpy matplotlib torch
```

---

In [ ]:
learning_objectives([
    "## Learning Objectives",
    "Quick-Start 00 and 01 (RL loop, Q-learning)",
    "Basic PyTorch (tensors, autograd, `optimizer.step()`)"
])

## Setup

We seed both NumPy and PyTorch. PyTorch's `manual_seed` covers CPU and GPU operations,
ensuring forward passes and weight initialisations are reproducible.

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Purpose: Imports and reproducibility
# Key Concept: PyTorch has its own RNG separate from NumPy — seed both

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

## From Q-Learning to Policy Gradients

Q-learning is **value-based**: learn $Q(s, a)$, then act greedily. It requires
discrete actions and struggles to generalise across states.

Policy gradient methods are **policy-based**: directly parameterise the policy
$\pi_\theta(a \mid s)$ and optimise $\theta$ via gradient ascent on expected return.

The **REINFORCE** update rule is:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} G_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \right]$$

Where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the **discounted return** from step $t$.

Intuition: if an action led to high return, increase its probability. If it led to low
return, decrease it. The gradient of the log-probability does this automatically.

In [ ]:
# Purpose: Define the policy network — maps state to action probabilities
# Key Concept: The network IS the policy: state in, probability distribution over actions out

class PolicyNetwork(nn.Module):
    """
    Two-layer MLP that maps a state observation to action probabilities.

    Architecture: Linear(obs_dim -> 64) -> ReLU -> Linear(64 -> 32) -> ReLU -> Linear(32 -> n_actions) -> Softmax

    Parameters
    ----------
    obs_dim : int
        Dimension of the observation space (4 for CartPole).
    n_actions : int
        Number of discrete actions (2 for CartPole).
    hidden_dim : int
        Width of both hidden layers.
    """

    def __init__(self, obs_dim: int, n_actions: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Parameters
        ----------
        x : torch.Tensor of shape (obs_dim,) or (batch, obs_dim)

        Returns
        -------
        torch.Tensor
            Action probabilities (softmax applied).
        """
        logits = self.net(x)
        return F.softmax(logits, dim=-1)

    def select_action(self, obs: np.ndarray):
        """
        Sample an action and return it with its log-probability.

        Parameters
        ----------
        obs : np.ndarray
            Single observation from the environment.

        Returns
        -------
        action : int
        log_prob : torch.Tensor (scalar, tracked for autograd)
        """
        obs_tensor = torch.FloatTensor(obs).to(DEVICE)
        probs = self.forward(obs_tensor)
        dist  = Categorical(probs)   # discrete probability distribution
        action = dist.sample()       # stochastic — key for exploration
        return action.item(), dist.log_prob(action)


# Verify shapes
env = gym.make('CartPole-v1')
obs_dim  = env.observation_space.shape[0]
n_actions = env.action_space.n

policy = PolicyNetwork(obs_dim=obs_dim, n_actions=n_actions).to(DEVICE)
sample_obs, _ = env.reset(seed=0)
action, log_prob = policy.select_action(sample_obs)

print(f"obs_dim={obs_dim}, n_actions={n_actions}")
print(f"Sample action: {action}, log_prob: {log_prob.item():.4f}")
print(f"Network parameters: {sum(p.numel() for p in policy.parameters()):,}")

## Computing Discounted Returns

After each episode we have a sequence of rewards $[r_0, r_1, \dots, r_T]$. For each
step $t$ we compute:

$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \dots$$

We then **normalise** $G_t$ (subtract mean, divide by std) to reduce variance.
This is a practical trick that stabilises REINFORCE without changing the direction
of the gradient update.

In [ ]:
# Purpose: Implement discounted return computation with normalisation
# Key Concept: Normalising returns reduces variance — one of the most impactful tricks in policy gradients

def compute_returns(rewards: list, gamma: float = 0.99) -> torch.Tensor:
    """
    Compute normalised discounted returns for an episode.

    Parameters
    ----------
    rewards : list of float
        Reward sequence from one episode.
    gamma : float
        Discount factor.

    Returns
    -------
    torch.Tensor of shape (T,)
        Normalised G_t values.
    """
    G = 0.0
    returns = []

    # Accumulate from the end backward (efficient one-pass computation)
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)

    returns = torch.FloatTensor(returns).to(DEVICE)

    # Normalise: zero mean, unit variance
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns


# Quick sanity check
test_rewards = [1.0, 1.0, 1.0, -10.0]
test_returns = compute_returns(test_rewards, gamma=0.99)
print("Rewards:          ", test_rewards)
print("Discounted returns:", [f"{v:.3f}" for v in test_returns.tolist()])
print("  (Negative at the end — poor final step is correctly penalised)")

## REINFORCE Training Loop

Each episode:
1. Roll out the policy to get a trajectory: $(s_0, a_0, r_0), (s_1, a_1, r_1), \dots$
2. Compute $G_t$ for every step
3. Compute the REINFORCE loss: $\mathcal{L} = -\sum_t G_t \log \pi_\theta(a_t \mid s_t)$
4. Backpropagate and update $\theta$

We train for 400 episodes — about 30-45 seconds on CPU.

In [ ]:
# Purpose: Full REINFORCE training loop
# Key Concept: The loss sign is negative because PyTorch minimises; we want to maximise return

GAMMA       = 0.99
LR          = 2e-3
N_EPISODES  = 400

torch.manual_seed(42)
np.random.seed(42)

env    = gym.make('CartPole-v1')
policy = PolicyNetwork(obs_dim=obs_dim, n_actions=n_actions, hidden_dim=64).to(DEVICE)
optimizer = optim.Adam(policy.parameters(), lr=LR)

scores = []
losses = []

for episode in range(N_EPISODES):
    obs, _ = env.reset(seed=episode)
    log_probs = []   # log π(a_t | s_t) for each step
    rewards   = []   # r_t for each step
    done = False

    # --- Step 1: Collect one episode of experience ---
    while not done:
        action, log_prob = policy.select_action(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        log_probs.append(log_prob)
        rewards.append(reward)

    # --- Step 2: Compute discounted, normalised returns ---
    returns = compute_returns(rewards, gamma=GAMMA)

    # --- Step 3: REINFORCE policy loss ---
    # Negative because we want gradient *ascent* on expected return
    policy_loss = -torch.stack(log_probs).mul(returns).sum()

    # --- Step 4: Backpropagate ---
    optimizer.zero_grad()
    policy_loss.backward()
    # Gradient clipping prevents exploding gradients
    torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
    optimizer.step()

    scores.append(sum(rewards))
    losses.append(policy_loss.item())

    if (episode + 1) % 100 == 0:
        recent_mean = np.mean(scores[-100:])
        print(f"Episode {episode+1:4d} | mean score (last 100): {recent_mean:6.1f} "
              f"| loss: {policy_loss.item():8.2f}")

env.close()
print("\nTraining complete.")

## Training Curves

Policy gradient methods are noisier than Q-learning because each update uses only a
single episode. The rolling average reveals the underlying trend.

In [ ]:
# Purpose: Visualise score and loss curves
# Key Concept: High variance in policy gradient methods is normal — look at the trend, not individual episodes

def rolling_mean(data, window=20):
    return np.convolve(data, np.ones(window) / window, mode='valid')


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: episode scores ---
ax = axes[0]
ep_range = np.arange(1, N_EPISODES + 1)
ax.plot(ep_range, scores, color='#e67e22', alpha=0.3, linewidth=0.8, label='Per-episode score')
roll = rolling_mean(scores, window=20)
ax.plot(np.arange(20, N_EPISODES + 1), roll, color='#d35400', linewidth=2.5,
        label='Rolling mean (20 ep)')
ax.axhline(200, color='#2ecc71', linestyle='--', linewidth=1.2, label='Score = 200')
ax.axhline(475, color='#27ae60', linestyle='--', linewidth=1.2, label='Score = 475 (solved)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Score (steps survived)', fontsize=12)
ax.set_title('REINFORCE Training Score', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Right: policy loss ---
ax = axes[1]
ax.plot(ep_range, losses, color='#8e44ad', alpha=0.4, linewidth=0.8)
ax.plot(np.arange(20, N_EPISODES + 1), rolling_mean(losses, window=20),
        color='#6c3483', linewidth=2.5, label='Rolling mean loss')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Policy Loss', fontsize=12)
ax.set_title('REINFORCE Loss', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('REINFORCE on CartPole-v1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Final 50-episode mean score: {np.mean(scores[-50:]):.1f}")
print(f"Peak score during training:  {max(scores)}")

## Clean Evaluation

Run 10 deterministic evaluation episodes. The policy is stochastic by design (it
outputs probabilities), but we can call `torch.argmax` instead of sampling to get the
greedy (highest-probability) action.

In [ ]:
# Purpose: Evaluate the learned policy deterministically
# Key Concept: Use argmax instead of sampling during evaluation for a fair performance estimate

policy.eval()   # disable dropout / batch-norm noise (none here, but good practice)

eval_env = gym.make('CartPole-v1')
eval_scores = []

with torch.no_grad():    # no gradient tracking needed during evaluation
    for test_ep in range(10):
        obs, _ = eval_env.reset(seed=2000 + test_ep)
        total = 0
        done  = False
        while not done:
            obs_tensor = torch.FloatTensor(obs).to(DEVICE)
            probs  = policy(obs_tensor)
            action = torch.argmax(probs).item()   # greedy, not sampled
            obs, reward, terminated, truncated, _ = eval_env.step(action)
            done  = terminated or truncated
            total += reward
        eval_scores.append(total)
        print(f"  Eval episode {test_ep+1:2d}: score = {int(total):3d}")

eval_env.close()
print(f"\nMean eval score: {np.mean(eval_scores):.1f} / 500")

In [ ]:
section_divider("Modify and Experiment")

## Modify and Experiment

These changes take only seconds to try:

1. **Remove return normalisation** in `compute_returns` — does training destabilise?
2. **Change `LR` to `1e-4`** — how much slower does learning become?
3. **Change `hidden_dim` to 16** — does a smaller network still learn?
4. **Remove gradient clipping** (`clip_grad_norm_`) — what happens in early episodes?
5. **Increase `N_EPISODES` to 800** — does the agent eventually reach score 475+?

## Key Takeaways

1. **REINFORCE directly optimises the policy** — no value function required
2. **The log-probability trick** lets us backpropagate through discrete action sampling
3. **Return normalisation** is a free variance-reduction step that almost always helps
4. **Gradient clipping** prevents catastrophic updates from high-variance episodes
5. **Policy gradient methods are noisier than Q-learning** but generalise better to
   continuous action spaces (e.g., robot joint angles)

## What's Next

REINFORCE has high variance because it uses full-episode returns. **Actor-Critic**
methods reduce this by learning a value function baseline alongside the policy.
**Quick-Start 03** builds DQN — a value-based deep RL approach that uses experience
replay and a target network for stable training. For full policy gradient theory, see
`modules/module_03_policy_gradients/` in this course.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])